In [ ]:
import os
import csv
import json
import glob
from collections import Counter
from transformers import AutoTokenizer

INPUT_DIR = "/kaggle/input/squad-v2"
train_candidates = glob.glob("/kaggle/input/**/train-v2.0.json", recursive=True)
dev_candidates = glob.glob("/kaggle/input/**/dev-v2.0.json", recursive=True)

TRAIN_PATH = train_candidates[0] if train_candidates else os.path.join(INPUT_DIR, "train-v2.0.json")
DEV_PATH = dev_candidates[0] if dev_candidates else os.path.join(INPUT_DIR, "dev-v2.0.json")
RESULTS_CSV_PATH = "/kaggle/working/results.csv"

print("TRAIN_PATH      :", TRAIN_PATH)
print("DEV_PATH        :", DEV_PATH)
print("RESULTS_CSV_PATH:", RESULTS_CSV_PATH)

def append_result_csv(experiment_name, model_name, max_length, stride, em=None, f1=None, train_time=None, notes=""):
    file_exists = os.path.isfile(RESULTS_CSV_PATH)
    fieldnames = ["experiment", "model", "max_length", "stride", "EM", "F1", "training_time_sec", "notes"]
    with open(RESULTS_CSV_PATH, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            "experiment": experiment_name,
            "model": model_name,
            "max_length": max_length,
            "stride": stride,
            "EM": em if em is not None else "",
            "F1": f1 if f1 is not None else "",
            "training_time_sec": train_time if train_time is not None else "",
            "notes": notes
        })

def load_squad(path, include_unanswerable=False):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    samples = []
    for article in raw["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if len(qa["answers"]) > 0:
                    ans = qa["answers"][0]
                    samples.append({
                        "id": qa["id"],
                        "question": qa["question"],
                        "context": context,
                        "answer_text": ans["text"],
                        "answer_start": ans["answer_start"],
                    })
                elif include_unanswerable:
                    samples.append({
                        "id": qa["id"],
                        "question": qa["question"],
                        "context": context,
                        "answer_text": "",
                        "answer_start": -1,
                    })
    return samples

train_samples = load_squad(TRAIN_PATH, include_unanswerable=False)
dev_samples = load_squad(DEV_PATH, include_unanswerable=False)

print(f"\nSố mẫu có đáp án - Train: {len(train_samples):,} | Dev: {len(dev_samples):,}")
print("Ví dụ mẫu 0:", train_samples[0])

TRAIN_PATH      : /kaggle/input/datasets/valphlien/squad-v2/train-v2.0.json
DEV_PATH        : /kaggle/input/datasets/valphlien/squad-v2/dev-v2.0.json
RESULTS_CSV_PATH: /kaggle/working/results.csv

Số mẫu có đáp án - Train: 86,821 | Dev: 5,928
Ví dụ mẫu 0: {'id': '56be85543aeaaa14008c9063', 'question': 'When did Beyonce start becoming popular?', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Ho

## Mục 10: Tokenization của QA (`input_ids`, `attention_mask`, `token_type_ids`)
Kiểm tra định dạng đầu vào `[CLS] question tokens [SEP] context tokens [SEP]` và sự khác biệt về `token_type_ids` giữa `bert-base-uncased` và `distilbert-base-uncased`.

In [7]:
question_ex = "What is the capital of France?"
context_ex = "The capital of France is Paris."

for model_name in ["bert-base-uncased", "distilbert-base-uncased"]:
    print("=" * 72)
    print(f"MODEL: {model_name}")
    tok = AutoTokenizer.from_pretrained(model_name)
    encoded = tok(question_ex, context_ex)

    print("Các keys trả về :", list(encoded.keys()))
    print("Tokens          :", tok.convert_ids_to_tokens(encoded["input_ids"]))
    print("input_ids       :", encoded["input_ids"])
    print("attention_mask  :", encoded["attention_mask"])

    if "token_type_ids" in encoded:
        print("token_type_ids  :", encoded["token_type_ids"], "--> Có sử dụng (0=Question, 1=Context)")
    else:
        print("token_type_ids  : KHÔNG CÓ (Model này không dùng token_type_ids)")

MODEL: bert-base-uncased
Các keys trả về : ['input_ids', 'token_type_ids', 'attention_mask']
Tokens          : ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
input_ids       : [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]
attention_mask  : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids  : [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1] --> Có sử dụng (0=Question, 1=Context)
MODEL: distilbert-base-uncased
Các keys trả về : ['input_ids', 'token_type_ids', 'attention_mask']
Tokens          : ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
input_ids       : [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]
attention_mask  : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids  :

## Mục 11 & Mục 12: Offset Mapping & Calculation Exercise 3
Ánh xạ character span sang token span với ví dụ chuẩn của Mục 12:
* **Context:** `"The capital of France is Paris."`
* **Answer:** `"Paris"`

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

question = "What is the capital of France?"
context = "The capital of France is Paris."
answer_text = "Paris"

# 1. Tính toán Character Span (Mục 12: ý 1, 2, 3)
char_start = context.index(answer_text)
char_end = char_start + len(answer_text)      # exclusive
char_last_inclusive = char_end - 1            # inclusive

print("--- KẾT QUẢ CALCULATION EXERCISE 3 (MỤC 12) ---")
print(f"Context gốc (độ dài {len(context)} ký tự): '{context}'")
print(f"1. Index ký tự đầu tiên ('{context[char_start]}'): {char_start}")
print(f"2. Index ký tự cuối cùng ('{context[char_last_inclusive]}'): {char_last_inclusive} (end exclusive = {char_end})")
print(f"3. Character span: ({char_start}, {char_end}) -> context[{char_start}:{char_end}] = '{context[char_start:char_end]}'")

# 2. Tokenize với return_offsets_mapping=True (Mục 11 & Mục 12: ý 4, 5, 6)
encoding = tokenizer(question, context, return_offsets_mapping=True)
input_ids = encoding["input_ids"]
offsets = encoding["offset_mapping"]
sequence_ids = encoding.sequence_ids()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("\n--- BẢNG ÁNH XẠ OFFSET MAPPING ---")
print(f"{'Token Idx':<10} {'Seq ID':<8} {'Token':<15} {'Offset (char)':<16} {'Substring gốc'}")
print("-" * 68)

start_token = None
end_token = None

for idx, (tok, (o_start, o_end), seq_id) in enumerate(zip(tokens, offsets, sequence_ids)):
    if seq_id == 0:
        sub = question[o_start:o_end]
    elif seq_id == 1:
        sub = context[o_start:o_end]
        if o_start <= char_start < o_end:
            start_token = idx
        if o_start < char_end <= o_end:
            end_token = idx
    else:
        sub = "<SPECIAL>"

    print(f"{idx:<10} {str(seq_id):<8} {tok:<15} {str((o_start, o_end)):<16} '{sub}'")

print("\n--- KẾT LUẬN TOKEN SPAN ---")
print(f"4. Token tương ứng của '{answer_text}': {tokens[start_token : end_token + 1]}")
print(f"5. start_token = {start_token}")
print(f"6. end_token   = {end_token}")
print(f"Decode kiểm chứng: '{tokenizer.decode(input_ids[start_token : end_token + 1])}'")

--- KẾT QUẢ CALCULATION EXERCISE 3 (MỤC 12) ---
Context gốc (độ dài 31 ký tự): 'The capital of France is Paris.'
1. Index ký tự đầu tiên ('P'): 25
2. Index ký tự cuối cùng ('s'): 29 (end exclusive = 30)
3. Character span: (25, 30) -> context[25:30] = 'Paris'

--- BẢNG ÁNH XẠ OFFSET MAPPING ---
Token Idx  Seq ID   Token           Offset (char)    Substring gốc
--------------------------------------------------------------------
0          None     [CLS]           (0, 0)           '<SPECIAL>'
1          0        what            (0, 4)           'What'
2          0        is              (5, 7)           'is'
3          0        the             (8, 11)          'the'
4          0        capital         (12, 19)         'capital'
5          0        of              (20, 22)         'of'
6          0        france          (23, 29)         'France'
7          0        ?               (29, 30)         '?'
8          None     [SEP]           (0, 0)           '<SPECIAL>'
9          1        th

## Mục 13: Truncation & Sliding Window
Kiểm tra cơ chế cắt context dài thành nhiều window gối nhau (`stride`) thông qua `return_overflowing_tokens=True`.

In [9]:
long_sample = max(train_samples[:5000], key=lambda s: len(s["context"]))

print("Question:", long_sample["question"])
print("Độ dài context (ký tự):", len(long_sample["context"]))
print("Answer:", long_sample["answer_text"], "| Vị trí char_start:", long_sample["answer_start"])

MAX_LEN_DEMO = 128
STRIDE_DEMO = 32

win_inputs = tokenizer(
    long_sample["question"],
    long_sample["context"],
    truncation="only_second",
    max_length=MAX_LEN_DEMO,
    stride=STRIDE_DEMO,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

num_windows = len(win_inputs["input_ids"])
print(f"\nSố window sinh ra (max_length={MAX_LEN_DEMO}, stride={STRIDE_DEMO}): {num_windows}")
print("overflow_to_sample_mapping:", win_inputs["overflow_to_sample_mapping"])

for w in range(num_windows):
    seq_ids = win_inputs.sequence_ids(w)
    offsets = win_inputs["offset_mapping"][w]
    ctx_offsets = [off for off, sid in zip(offsets, seq_ids) if sid == 1]
    c_start, c_end = ctx_offsets[0][0], ctx_offsets[-1][1]
    has_ans = (c_start <= long_sample["answer_start"]) and (long_sample["answer_start"] + len(long_sample["answer_text"]) <= c_end)
    print(f"  Window {w}: bao phủ context char [{c_start:>4} : {c_end:>4}] | Chứa trọn vẹn answer? -> {has_ans}")

Question: Which journalist considered Spectre the worst James Bond movie in three decades?
Độ dài context (ký tự): 3076
Answer: Scott Mendelson | Vị trí char_start: 515

Số window sinh ra (max_length=128, stride=32): 9
overflow_to_sample_mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0]
  Window 0: bao phủ context char [   0 :  458] | Chứa trọn vẹn answer? -> False
  Window 1: bao phủ context char [ 338 :  806] | Chứa trọn vẹn answer? -> True
  Window 2: bao phủ context char [ 672 : 1129] | Chứa trọn vẹn answer? -> False
  Window 3: bao phủ context char [ 994 : 1500] | Chứa trọn vẹn answer? -> False
  Window 4: bao phủ context char [1345 : 1843] | Chứa trọn vẹn answer? -> False
  Window 5: bao phủ context char [1719 : 2144] | Chứa trọn vẹn answer? -> False
  Window 6: bao phủ context char [2028 : 2507] | Chứa trọn vẹn answer? -> False
  Window 7: bao phủ context char [2359 : 2831] | Chứa trọn vẹn answer? -> False
  Window 8: bao phủ context char [2698 : 3076] | Chứa trọn vẹn answer? -> False


## Mục 16: Answer Span Alignment
Ánh xạ `(answer_start, answer_end)` sang `(start_token, end_token)`, xử lý các trường hợp biên (answer bắt đầu/kết thúc giữa token, answer bị truncation hoặc nằm ngoài window), và lưu thống kê alignment vào `results.csv`.

In [10]:
def align_answer_spans(tokenizer, samples, max_length=384, stride=128):
    questions = [s["question"] for s in samples]
    contexts = [s["context"] for s in samples]

    tokenized = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized["overflow_to_sample_mapping"]
    offset_mapping = tokenized["offset_mapping"]

    start_positions = []
    end_positions = []
    stats = {
        "total_windows": len(offset_mapping),
        "valid_answer_windows": 0,
        "out_of_window_or_truncated": 0,
        "subword_boundary_mismatch": 0,
    }

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)

        sample_idx = sample_mapping[i]
        answer_start = samples[sample_idx]["answer_start"]
        answer_text = samples[sample_idx]["answer_text"]

        if answer_start == -1 or len(answer_text) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            stats["out_of_window_or_truncated"] += 1
            continue

        answer_end = answer_start + len(answer_text)

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Answer nằm ngoài window hoặc bị truncation cắt ngang
        if not (offsets[token_start_index][0] <= answer_start and offsets[token_end_index][1] >= answer_end):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            stats["out_of_window_or_truncated"] += 1
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= answer_start:
                token_start_index += 1
            st = token_start_index - 1

            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1
            en = token_end_index + 1

            start_positions.append(st)
            end_positions.append(en)
            stats["valid_answer_windows"] += 1

            if offsets[st][0] != answer_start or offsets[en][1] != answer_end:
                stats["subword_boundary_mismatch"] += 1

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized, stats

# Chạy alignment trên 2,000 mẫu đầu tiên và ghi log vào results.csv
subset = train_samples[:2000]
features, alignment_stats = align_answer_spans(tokenizer, subset, max_length=384, stride=128)

print("=== THỐNG KÊ ALIGNMENT TRÊN 2,000 MẪU ===")
for k, v in alignment_stats.items():
    print(f"- {k}: {v}")

# Ghi nối tiếp kết quả kiểm tra alignment vào results.csv
note_str = (
    f"Alignment check (2000 samples): total_windows={alignment_stats['total_windows']}, "
    f"valid={alignment_stats['valid_answer_windows']}, "
    f"truncated/out={alignment_stats['out_of_window_or_truncated']}, "
    f"subword_mismatch={alignment_stats['subword_boundary_mismatch']}"
)
append_result_csv(
    experiment_name="Tokenization_Alignment_Check",
    model_name="bert-base-uncased",
    max_length=384,
    stride=128,
    notes=note_str
)

=== THỐNG KÊ ALIGNMENT TRÊN 2,000 MẪU ===
- total_windows: 2050
- valid_answer_windows: 2009
- out_of_window_or_truncated: 41
- subword_boundary_mismatch: 12


In [11]:
print("--- KIỂM TRA CÁC TRƯỜNG HỢP BIÊN (MỤC 16) ---")

# Case 1 & 2: Answer bắt đầu hoặc kết thúc giữa token (do subword split)
found_subword_case = False
for w_idx in range(len(features["input_ids"])):
    st = features["start_positions"][w_idx]
    en = features["end_positions"][w_idx]
    cls_idx = features["input_ids"][w_idx].index(tokenizer.cls_token_id)
    if st == cls_idx:
        continue

    s_idx = features["overflow_to_sample_mapping"][w_idx]
    ans_start = subset[s_idx]["answer_start"]
    ans_end = ans_start + len(subset[s_idx]["answer_text"])
    offsets = features["offset_mapping"][w_idx]

    if offsets[st][0] < ans_start or offsets[en][1] > ans_end:
        tokens = tokenizer.convert_ids_to_tokens(features["input_ids"][w_idx])
        print("[Case 1 & 2: Answer bắt đầu/kết thúc giữa token]")
        print(f"  Sample {s_idx} | Gold answer: '{subset[s_idx]['answer_text']}' (char {ans_start}->{ans_end})")
        print(f"  Mapped tokens: {tokens[st : en + 1]} (char {offsets[st][0]}->{offsets[en][1]})")
        print(f"  Decoded span : '{tokenizer.decode(features['input_ids'][w_idx][st : en + 1])}'\n")
        found_subword_case = True
        break

if not found_subword_case:
    print("[Case 1 & 2]: Không có subword boundary mismatch trong tập mẫu kiểm tra.\n")

# Case 3 & 4: Answer bị truncation hoặc nằm ngoài window
win_counts = Counter(features["overflow_to_sample_mapping"])
multi_win_samples = [s_idx for s_idx, cnt in win_counts.items() if cnt > 1]

if multi_win_samples:
    target_s = multi_win_samples[0]
    print(f"[Case 3 & 4: Answer nằm ngoài window hoặc bị truncation] (Sample {target_s})")
    print(f"  Gold answer: '{subset[target_s]['answer_text']}' (char_start={subset[target_s]['answer_start']})")
    for w_idx, s_idx in enumerate(features["overflow_to_sample_mapping"]):
        if s_idx == target_s:
            st = features["start_positions"][w_idx]
            en = features["end_positions"][w_idx]
            decoded = tokenizer.decode(features["input_ids"][w_idx][st : en + 1])
            print(f"  -> Window {w_idx}: start_pos={st}, end_pos={en} | Decoded='{decoded}'")

--- KIỂM TRA CÁC TRƯỜNG HỢP BIÊN (MỤC 16) ---
[Case 1 & 2: Answer bắt đầu/kết thúc giữa token]
  Sample 92 | Gold answer: 'split with Luckett and Rober' (char 194->222)
  Mapped tokens: ['split', 'with', 'luck', '##ett', 'and', 'robe', '##rson'] (char 194->225)
  Decoded span : 'split with luckett and roberson'

[Case 3 & 4: Answer nằm ngoài window hoặc bị truncation] (Sample 173)
  Gold answer: 'Jay Z' (char_start=34)
  -> Window 173: start_pos=18, end_pos=19 | Decoded='jay z'
  -> Window 174: start_pos=0, end_pos=0 | Decoded='[CLS]'
